# Modelo de Predicción Team Attributes

MODELO XGBOOST

TARGET: Home Win (H), Draw (D), Away Win (A)

FEATURES: Team Attributes

**Cargamos los Dataframes Necesarios**

In [1]:
# HELPERS
import sqlite3 as sql
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

Match

In [2]:
conn = sql.connect('../../data/database.sqlite')
query = "SELECT * FROM Match"
df_match = pd.read_sql_query(query, conn)

Se crea una columna Full Time Results (FTR) con las clases Home Win (H), Draw (D), Away Win (A) numerizadas (0, 1 y 2 respectivamente).

In [3]:
conditions = [
    df_match["home_team_goal"] > df_match["away_team_goal"],
    df_match["home_team_goal"] < df_match["away_team_goal"]
]

choices = [0, 2]

df_match["FTR"] = np.select(conditions, choices, default=1) # type: ignore

Se definen las columnas con proveedores de casas de apuestas para eliminarlas del dataframe de partidos

In [4]:
betting_columns = [
    'B365H', 'B365D', 'B365A', 'BWH', 'BWD', 'BWA', 
    'IWH', 'IWD', 'IWA', 'LBH', 'LBD', 'LBA', 
    'PSH', 'PSD', 'PSA', 'WHH', 'WHD', 'WHA', 
    'SJH', 'SJD', 'SJA', 'VCH', 'VCD', 'VCA', 
    'GBH', 'GBD', 'GBA', 'BSH', 'BSD', 'BSA'
]
df_match = df_match.drop(columns=betting_columns)

Se eliminan el resto de columnas con datos nominales

In [5]:
df_match = df_match.select_dtypes(include="number")

Team_Attributes

In [6]:
conn = sql.connect('../../data/database.sqlite')
query = "SELECT * FROM Team_Attributes"
df_team_attributes = pd.read_sql_query(query, conn)

Se obtiene la última versión actualizada con los datos del equipo. Se eliminan los atributos nominales para quedarnos con los numéricos, es decir sus estadísticas tácticas como equipo.

In [7]:
team_attrs = (
    df_team_attributes
    .sort_values("date")
    .drop_duplicates("team_api_id", keep="last")
    .select_dtypes(include="number")
)
df_teams = df_match[["match_api_id", "FTR"]].copy()

Se identifican las columnas del partido que contienen el ID del equipo (`home_team_api_id`, `away_team_api_id`), excluyendo explícitamente `home_team_goal`/`away_team_goal` ya que estas determinan directamente el target (FTR) y usarlas como feature sería data leakage.

In [8]:
import re

team_cols = [c for c in df_match.columns if re.match(r"(home|away)_team", c) and c not in ["home_team_goal", "away_team_goal"]]
print(f"Columnas de equipos encontradas: {len(team_cols)}")
team_cols

Columnas de equipos encontradas: 2


['home_team_api_id', 'away_team_api_id']

Del dataframe `team_attrs` se descartan los identificadores (`id`, `team_fifa_api_id`) que no aportan información como feature, quedando solo `team_api_id` (clave para el join) y las estadísticas tácticas numéricas del equipo.

In [9]:
stat_cols = [c for c in team_attrs.columns if c not in ("id", "team_fifa_api_id", "team_api_id")]
stat_cols

['buildUpPlaySpeed',
 'buildUpPlayDribbling',
 'buildUpPlayPassing',
 'chanceCreationPassing',
 'chanceCreationCrossing',
 'chanceCreationShooting',
 'defencePressure',
 'defenceAggression',
 'defenceTeamWidth']

Se define una función auxiliar que, dado un ID de equipo (columna del partido), devuelve sus estadísticas numéricas con el nombre de columna prefijado según el rol (`home_team_api_id_*` / `away_team_api_id_*`). El `reset_index` antes del merge y la restauración del índice original al final garantizan que el orden de las filas se preserve exactamente.

In [10]:
def get_team_stats(id_series, prefix):
    """
    Devuelve las estadisticas numericas mas recientes del equipo identificado en id_series,
    con las columnas prefijadas segun la posicion (ej: home_team_1_overall_rating).
    """
    tmp = id_series.reset_index(drop=True).to_frame(name="team_api_id")
    tmp = tmp.merge(team_attrs[["team_api_id"] + stat_cols], on="team_api_id", how="left")
    tmp = tmp.drop(columns="team_api_id")
    tmp.columns = [f"{prefix}_{c}" for c in tmp.columns]
    tmp.index = id_series.index
    return tmp

Se construye el dataframe final de features concatenando `match_api_id`, `FTR` y las estadísticas tácticas de ambos equipos.

In [11]:
feature_blocks = [get_team_stats(df_match[col], col) for col in team_cols]

df_teams = pd.concat([df_match[["match_api_id", "FTR"]]] + feature_blocks, axis=1)
df_teams.shape

(25979, 20)

Se revisa la proporción de valores faltantes por columna. A diferencia del caso de jugadores, acá la causa más común no es que falte el registro del equipo, sino que el atributo `buildUpPlayDribbling` fue incorporado más tarde a la base de FIFA: para temporadas anteriores a su inclusión, el valor queda como nulo aunque el resto de los atributos del equipo sí estén presentes. Por eso esa columna concentra la mayor proporción de faltantes.

In [12]:
missing_ratio = df_teams.isna().mean().sort_values(ascending=False)
missing_ratio.head(10)

away_team_api_id_buildUpPlayDribbling      0.052850
home_team_api_id_buildUpPlayDribbling      0.052812
home_team_api_id_defenceTeamWidth          0.006852
away_team_api_id_buildUpPlaySpeed          0.006852
away_team_api_id_defenceAggression         0.006852
away_team_api_id_defencePressure           0.006852
away_team_api_id_chanceCreationShooting    0.006852
away_team_api_id_chanceCreationCrossing    0.006852
away_team_api_id_chanceCreationPassing     0.006852
away_team_api_id_buildUpPlayPassing        0.006852
dtype: float64

Se imputan los nulos con la mediana de cada columna, conservando los valores usados (`feature_medians`) para poder reutilizarlos en una eventual inferencia posterior, sin tener que recalcularlos sobre datos parciales.

In [13]:
feature_columns = [c for c in df_teams.columns if c not in ("match_api_id", "FTR")]

feature_medians = df_teams[feature_columns].median()
df_teams[feature_columns] = df_teams[feature_columns].fillna(feature_medians)

**División en conjuntos de entrenamiento y prueba**

In [14]:
X = df_teams[feature_columns]
y = df_teams["FTR"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

**Entrenamiento del modelo XGBoost**

El parámetro correcto para indicar la cantidad de clases en un problema multiclase es `num_class`, no `num_classes`; este último es ignorado silenciosamente por XGBoost (de ahí el `UserWarning: Parameters: { "num_classes" } are not used` que aparecía al entrenar).

In [15]:
model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    tree_method="approx",
    random_state=42
)

model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

**Evaluación del modelo**

In [16]:
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Home Win", "Draw", "Away Win"]))

Accuracy: 0.5037

              precision    recall  f1-score   support

    Home Win       0.51      0.88      0.65      2384
        Draw       0.28      0.02      0.03      1319
    Away Win       0.49      0.34      0.40      1493

    accuracy                           0.50      5196
   macro avg       0.43      0.41      0.36      5196
weighted avg       0.45      0.50      0.42      5196



La matriz de confusión permite ver, fila por fila, en qué clase real (Home/Draw/Away) el modelo acierta o se confunde, y con qué clase predicha confunde más frecuentemente cada caso.

In [17]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

print(f"Accuracy (train): {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Accuracy (test):  {accuracy_score(y_test, y_pred_test):.4f}")

Accuracy (train): 0.5301
Accuracy (test):  0.5037


In [22]:
test_report = classification_report(
    y_test, y_pred_test, target_names=["Home Win", "Draw", "Away Win"], output_dict=True
)
df_test_report = pd.DataFrame(test_report).transpose().round(2)
df_test_report

,precision,recall,f1-score,support
Home Win,0.51,0.88,0.65,2384.0
Draw,0.28,0.02,0.03,1319.0
Away Win,0.49,0.34,0.40,1493.0
accuracy,0.50,0.50,0.50,0.5
macro avg,0.43,0.41,0.36,5196.0
weighted avg,0.45,0.50,0.42,5196.0


In [24]:
print(df_test_report)

              precision  recall  f1-score  support
Home Win           0.51    0.88      0.65   2384.0
Draw               0.28    0.02      0.03   1319.0
Away Win           0.49    0.34      0.40   1493.0
accuracy           0.50    0.50      0.50      0.5
macro avg          0.43    0.41      0.36   5196.0
weighted avg       0.45    0.50      0.42   5196.0


In [23]:
train_report = classification_report(
    y_train, y_pred_train, target_names=["Home Win", "Draw", "Away Win"], output_dict=True
)
df_train_report = pd.DataFrame(train_report).transpose().round(2)
df_train_report

,precision,recall,f1-score,support
Home Win,0.52,0.91,0.66,9533.00
Draw,0.64,0.03,0.06,5277.00
Away Win,0.55,0.37,0.44,5973.00
accuracy,0.53,0.53,0.53,0.53
macro avg,0.57,0.44,0.39,20783.00
weighted avg,0.56,0.53,0.45,20783.00


In [20]:
confusion_matrix(y_test, y_pred)

array([[2086,   27,  271],
       [1033,   21,  265],
       [ 955,   28,  510]])

**Importancia de features**

`feature_importances_` mide cuánto contribuyó cada feature, en promedio, a reducir el error de predicción a lo largo de todos los árboles del ensamble.

In [21]:
importances = pd.Series(model.feature_importances_, index=feature_columns)
importances.sort_values(ascending=False).head(20)

away_team_api_id_defencePressure           0.101250
home_team_api_id_defenceAggression         0.076299
away_team_api_id_defenceAggression         0.074673
home_team_api_id_defencePressure           0.073337
away_team_api_id_buildUpPlayPassing        0.065909
home_team_api_id_chanceCreationShooting    0.056874
home_team_api_id_buildUpPlayPassing        0.056424
home_team_api_id_buildUpPlayDribbling      0.056245
away_team_api_id_defenceTeamWidth          0.054282
home_team_api_id_chanceCreationPassing     0.052001
away_team_api_id_buildUpPlayDribbling      0.049094
away_team_api_id_chanceCreationPassing     0.047372
home_team_api_id_defenceTeamWidth          0.047061
away_team_api_id_chanceCreationShooting    0.046261
away_team_api_id_buildUpPlaySpeed          0.042605
away_team_api_id_chanceCreationCrossing    0.042391
home_team_api_id_chanceCreationCrossing    0.029625
home_team_api_id_buildUpPlaySpeed          0.028299
dtype: float32